In [109]:
import numpy as np
import pandas as pd

In [110]:
# データを読み込みます
df = pd.read_csv('../Job.csv',parse_dates=['EVENT_DTM'],dtype={'SERIAL_NUMBER':'category', 'PRINT_COUNT':'int32'})

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   SERIAL_NUMBER  13 non-null     category      
 1   EVENT_DTM      13 non-null     datetime64[ns]
 2   PRINT_COUNT    13 non-null     int32         
dtypes: category(1), datetime64[ns](1), int32(1)
memory usage: 313.0 bytes


In [112]:
df['JOB_INTERVAL'] = df.EVENT_DTM.diff().map(lambda x: x.total_seconds())
df['JOB_LENGTH'] = df.PRINT_COUNT.diff()
df['SERIAL_NUMBER_SHIFT'] = df.SERIAL_NUMBER.shift()
df['CONTINUE'] = df.JOB_INTERVAL < df.JOB_LENGTH/45*60+2
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'JOB_INTERVAL'] = np.nan
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'JOB_LENGTH'] = np.nan
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'CONTINUE'] = False

In [113]:
df

,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_INTERVAL,JOB_LENGTH,SERIAL_NUMBER_SHIFT,CONTINUE
0,aaa,2020-01-01 15:15:00,5000,NaN,NaN,NaN,False
1,aaa,2020-01-02 15:15:12,5001,86412.0,1.0,aaa,False
2,aaa,2020-01-02 15:15:13,5002,1.0,1.0,aaa,True
3,aaa,2020-01-02 15:15:14,5004,1.0,2.0,aaa,True
4,aaa,2020-01-01 15:16:00,5006,-86354.0,2.0,aaa,True
5,aaa,2020-01-02 15:17:12,5007,86472.0,1.0,aaa,False
6,aaa,2020-01-02 15:18:13,5008,61.0,1.0,aaa,False
7,aaa,2020-01-03 15:18:14,5009,86401.0,1.0,aaa,False
8,aaa,2020-01-04 15:18:15,5010,86401.0,1.0,aaa,False
9,aaa,2020-01-05 15:18:16,5011,86401.0,1.0,aaa,False


In [114]:
# 連続JOB開始行のインデックスと連続JOB枚数の辞書作成
value_list = []
index_list = []
continue_dict = {}
for i, (JOB_LENGTH, CONTINUE) in enumerate(zip(df.JOB_LENGTH.values, df.CONTINUE.values)):
    if CONTINUE == False and len(index_list) == 0:
        JOB_LENGTH_BEFOR = 0
    elif CONTINUE == False and len(index_list) > 0:
        index_min = min(index_list)
        value_max = max(value_list)
        continue_dict[index_min] = value_max
        value_list = []
        index_list = []
    else:
        JOB_LENGTH += JOB_LENGTH_BEFOR
        JOB_LENGTH_BEFOR = JOB_LENGTH
        value_list.append(JOB_LENGTH)
        index = i-1
        index_list.append(index)
    
# 辞書から新しい列作成
for k, v in continue_dict.items():
    df.loc[df.index[k],'ADJUST'] = v

# 連続JOBを考慮したJOB枚数を計算
df.loc[~df.ADJUST.isnull(), 'JOB_LENGTH'] = df.JOB_LENGTH + df.ADJUST

In [115]:
df


,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_INTERVAL,JOB_LENGTH,SERIAL_NUMBER_SHIFT,CONTINUE,ADJUST
0,aaa,2020-01-01 15:15:00,5000,NaN,NaN,NaN,False,NaN
1,aaa,2020-01-02 15:15:12,5001,86412.0,6.0,aaa,False,5.0
2,aaa,2020-01-02 15:15:13,5002,1.0,1.0,aaa,True,NaN
3,aaa,2020-01-02 15:15:14,5004,1.0,2.0,aaa,True,NaN
4,aaa,2020-01-01 15:16:00,5006,-86354.0,2.0,aaa,True,NaN
5,aaa,2020-01-02 15:17:12,5007,86472.0,1.0,aaa,False,NaN
6,aaa,2020-01-02 15:18:13,5008,61.0,1.0,aaa,False,NaN
7,aaa,2020-01-03 15:18:14,5009,86401.0,1.0,aaa,False,NaN
8,aaa,2020-01-04 15:18:15,5010,86401.0,1.0,aaa,False,NaN
9,aaa,2020-01-05 15:18:16,5011,86401.0,1.0,aaa,False,NaN


In [122]:
def aaa(row):
    print(row[0])
    print(row['SERIAL_NUMBER'])
    if row[0].startswith('aa'):
        # print(row[1]+10)
        return (row[1]+10)
    else:
        # print(row[1]+100)
        return (row[1]+100)

In [123]:
# df[['A','B']] = df.apply(aaa,axis=1, result_type='expand')
# df['A'] = df[['PRINT_COUNT','JOB_LENGTH']].apply(aaa,axis=1, result_type='expand')
df['A'] = df[['SERIAL_NUMBER','JOB_LENGTH']].apply(aaa,axis=1)
# df['B'] = df[['SERIAL_NUMBER','JOB_LENGTH']].apply(aaa,axis=1)

aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
aaa
bbb
bbb
bbb
bbb
bbb
bbb


In [102]:
df

,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_INTERVAL,JOB_LENGTH,SERIAL_NUMBER_SHIFT,CONTINUE,ADJUST,A,B
0,aaa,2020-01-01 15:15:00,5000,NaN,NaN,NaN,False,NaN,NaN,NaN
1,aaa,2020-01-02 15:15:12,5001,86412.0,6.0,aaa,False,5.0,16.0,86422.0
2,aaa,2020-01-02 15:15:13,5002,1.0,1.0,aaa,True,NaN,11.0,11.0
3,aaa,2020-01-02 15:15:14,5004,1.0,2.0,aaa,True,NaN,12.0,11.0
4,aaa,2020-01-01 15:16:00,5006,-86354.0,2.0,aaa,True,NaN,12.0,-86344.0
5,aaa,2020-01-02 15:17:12,5007,86472.0,1.0,aaa,False,NaN,11.0,86482.0
6,aaa,2020-01-02 15:18:13,5008,61.0,1.0,aaa,False,NaN,11.0,71.0
7,aaa,2020-01-03 15:18:14,5009,86401.0,1.0,aaa,False,NaN,11.0,86411.0
8,aaa,2020-01-04 15:18:15,5010,86401.0,1.0,aaa,False,NaN,11.0,86411.0
9,aaa,2020-01-05 15:18:16,5011,86401.0,1.0,aaa,False,NaN,11.0,86411.0


In [617]:
# 連続Job行の削除
df = df[df.CONTINUE == False]
# 不要列の削除
df = df.drop(['CONTINUE','ADJUST','JOB_INTERVAL'], axis=1)
# JOB間隔の再計算（単位：分）
df['JOB_INTERVAL'] = round(df.EVENT_DTM.diff().map(lambda x: x.total_seconds()/60), 3)
df.loc[df.SERIAL_NUMBER != df.SERIAL_NUMBER_SHIFT, 'JOB_INTERVAL'] = np.nan
# 不要列の削除
df = df.drop(['SERIAL_NUMBER_SHIFT'], axis=1)
df = df.dropna(subset=['JOB_LENGTH'])

In [618]:
df

,SERIAL_NUMBER,EVENT_DTM,PRINT_COUNT,JOB_LENGTH,JOB_INTERVAL
1,aaa,2020-01-02 15:15:12,5001,6.0,1440.200
5,aaa,2020-01-02 15:17:12,5007,1.0,2.000
6,aaa,2020-01-02 15:18:13,5008,1.0,1.017
7,aaa,2020-01-03 15:18:14,5009,1.0,1440.017
8,aaa,2020-01-04 15:18:15,5010,1.0,1440.017
9,aaa,2020-01-05 15:18:16,5011,1.0,1440.017
12,bbb,2020-01-03 15:15:15,5020,7.0,1440.033


In [268]:
def fuser_Cycle(df):
    # 並び替え
    df = df.sort_values(['Deviceid', 'Timestamp']).reset_index(drop = True)
    # 列追加
    df['Unit'] = 'Fuser'
    df['Deviceid_Shift'] = df.Deviceid.shift()
    df['Life_Diff'] = df['Life'].diff()
    df['JOB_LENGTH'] = df['Total Page Count'].diff()
    df['Rol_Diff'] = df['Total Rol Distance'].diff()
    # Deviceidの切替りのデータをnanに置き換える
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Life_Diff'] = np.nan
    df.loc[df.Deviceid != df.Deviceid_Shift, 'JOB_LENGTH'] = np.nan
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Rol_Diff'] = np.nan
    # 交換タイミングを計算
    df.loc[(df.Life_Diff > 0) & (df.JOB_LENGTH < 0) & (df.Rol_Diff < 0), 'Event'] = '交換'
    # 不要な列を削除
    df = df.drop(['Deviceid_Shift', 'Life_Diff', 'JOB_LENGTH', 'Rol_Diff'], axis=1)
    return df

In [269]:
def life_cycle(df):
    for row in range(len(df)):
        if row == 0:
            cycle = 1
            count_new = 0
            df.loc[row,'Cycle'] = cycle
            df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount']
        else:
            # 同じ本体、新品以外
            if df.loc[row,'Event'] != '交換' and df.loc[row,'Deviceid'] == df.loc[row-1,'Deviceid']:
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
            # 同じ本体、新品
            elif df.loc[row,'Event'] == '交換' and df.loc[row,'Deviceid'] == df.loc[row-1,'Deviceid']:
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
                # 2サイクル目以降で交換後1000枚以内の再交換はnanに置き換える
                if cycle == 1:
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                elif df.loc[row,'Pagecount'] - count_new > 1000 :
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                else:
                    df.loc[row,'Event'] = np.nan
            # 違う本体への切替り、新品以外
            elif df.loc[row,'Event'] != '交換' and df.loc[row,'Deviceid'] != df.loc[row-1,'Deviceid']:
                cycle = 1
                count_new = 0
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
            # 違う本体への切替り、新品
            elif df.loc[row,'Event'] == '交換' and df.loc[row,'Deviceid'] != df.loc[row-1,'Deviceid']:
                cycle = 1
                count_new = 0
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
                # 交換後1000枚以内の再交換はnanに置き換える
                if cycle == 1:
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                elif df.loc[row,'Pagecount'] - count_new > 1000 :
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                else:
                    df.loc[row,'Event'] = np.nan
    return df

In [270]:
# 交換タイミングの取得
if apps == 'fuser_life':
    df = fuser_Cycle(df) 
elif apps == 'itb_life':
    df = fuser_Cycle(df) 

# 交換サイクルとリセットページカウントの算出
df = life_cycle(df)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,Fuser,NaN,1.0,5000.0


In [271]:
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,Fuser,NaN,1.0,5000.0


In [272]:
df['Life2'] = df.Life.apply(lambda x : x if x<5 else np.nan)

In [273]:
df = df.sort_values(['Deviceid', 'Cycle']).reset_index(drop = True)
df.groupby(['Deviceid','Cycle']).max()

/var/folders/gx/lc33vdvs5dj0sf9x1njqzryr0000gn/T/ipykernel_35050/807355777.py:2: FutureWarning: Dropping invalid columns in DataFrameGroupBy.max is deprecated. In a future version, a TypeError will be raised. Before calling .max, select only columns which should be valid for the function.
  df.groupby(['Deviceid','Cycle']).max()


Timestamp  Pagecount  Life  Total Page Count  \
Deviceid Cycle                                                 
aaa      1.0   2020-01-04      13000   100             12000   
         2.0   2020-01-06      15000   100              4000   
bbb      1.0   2020-01-08          8   100                 2   
ccc      1.0   2020-01-18      15000   100              4500   
         2.0   2020-01-21      15200   100             15000   

                Total Rol Distance   Unit  Reset_Count  Life2  
Deviceid Cycle                                                 
aaa      1.0                 12000  Fuser      13000.0    1.0  
         2.0                  4000  Fuser       2000.0    NaN  
bbb      1.0                 40000  Fuser          8.0    NaN  
ccc      1.0                  4500  Fuser      15000.0   -8.0  
         2.0                 15000  Fuser        200.0   -8.0

In [274]:
# 各デバイス毎の一番最初と最後のデータのみの日付データを作成（エラー発生なし本体の計算と、各本体のデータ取得期間確認用）
# df_serial1 = pd.DataFrame()
# df_serial2 = pd.DataFrame()
# df_serial1['start'] = df.groupby('serial').start.min()
# df_serial2['start'] = df.groupby('serial').start.max()
# df_serial = pd.concat([df_serial1,df_serial2])
# df_serial = df_serial.reset_index()
# df_serial = df_serial.sort_values(['serial','start'])
# df_serial


In [275]:
# 各デバイス毎の一番最初と最後のデータのみを抽出（エラー発生なし本体の計算と、各本体のデータ取得期間確認用）
df_group = df.groupby('serial')
df_serial1 = df.loc[df_group['start'].idxmin(),:]
df_serial2 = df.loc[df_group['start'].idxmax(),:]
df_serial = pd.concat([df_serial1,df_serial2])
df_serial = df_serial.sort_values(['serial','start'])
df_serial = df_serial.reset_index(drop=True)
df_serial

KeyError: 'serial'

In [250]:
# エラー発生時のインデックスリストを作成
index_list = df[~df.factor.isnull()].index.to_list()
index_list

AttributeError: 'DataFrame' object has no attribute 'factor'

In [276]:
# エラー発生前後50行のインデックスリストを作成
index_max = len(df)
hold_list = []
for i in index_list:
    for j in range(-50,51):
        if i+j>=0 and i+j<index_max:
            hold_list.append(i+j)
hold_list = list(set(hold_list))
hold_list

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

In [252]:
# エラー発生前後50行のデータのみを残す
df.loc[df.index[hold_list],'Hold'] = 1
df = df.dropna(subset=['Hold'])
df = df.drop('Hold',axis=1)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Life2,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,NaN,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,1.0,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,-5.0,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,NaN,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,NaN,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,NaN,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,NaN,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,NaN,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,NaN,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,-8.0,Fuser,NaN,1.0,5000.0


In [253]:
df = pd.concat([df,df_serial])
df = df.sort_values(['serial','start'])
df = df.reset_index(drop=True)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Life2,Unit,Event,Cycle,Reset_Count,start,end,nr,serial,error_code,factor,print_count
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1.0
1,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-06,2020-03-20,600-1,aaa,2.0,NaN,6.0
2,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-07,2020-01-07,701-1,bbb,4.0,NaN,7.0
3,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-15,2020-01-07,801-1,bbb,4.0,JAM,15.0
4,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-16,2020-01-07,801-1,ccc,8.0,JAM,16.0
5,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-17,2020-01-07,801-1,ccc,1.0,JAM,17.0
6,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-18,2020-02-07,600-1,eee,NaN,NaN,18.0
7,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-24,2020-01-07,801-1,eee,NaN,JAM,24.0
8,aaa,2020-01-01,4000.0,74.0,4000.0,4000.0,NaN,Fuser,NaN,1.0,4000.0,NaT,NaT,NaN,NaN,NaN,NaN,NaN
9,aaa,2020-01-02,8000.0,1.0,8000.0,8000.0,1.0,Fuser,NaN,1.0,8000.0,NaT,NaT,NaN,NaN,NaN,NaN,NaN


In [4]:
print("This is a \"quote\" inside a string.")  # ダブルクォートのエスケープ
print('This is an \'example\' of single quotes.')  # シングルクォートのエスケープ
print("C:\\path\\to\\file")  # バックスラッシュのエスケープ
print("Hello\nWorld")  # 改行文字のエスケープ
print("Tab\tTab\tTab")  # タブ文字のエスケープ


This is a "quote" inside a string.
This is an 'example' of single quotes.
C:\path\to\file
Hello
World
Tab	Tab	Tab


In [ ]:
プログラムコード生成を題材に、Copilotの実用的な使い方をまとめました
Copilotを使えば、プログラミング初心者でもコードが作成できるようになります


■コード生成プロンプト作成のコツ
このようなフォーマットで指示することで、様々なコード生成に活用できます
（Copilotの入力画面で説明）
役割と指示を与える
フォルダ構成、ファイル名、データの中身等について記載する
処理内容の要件を記載する

■紹介する事例
以下の３種類のコード生成の事例を紹介します
VBA：エクセル処理
Python：データ分析
batファイル：ファイル操作/Python実行

■VBAのコード生成事例
複数のcsvファイルに処理をして、一つのエクセルファイルにまとめる処理を自動化する事例です

■VBA コード作成手順
【プロンプト】
あなたは優秀なVBAのプログラマです。
以下の #要件 に合うVBAのコードを作成してください。

#前提条件
フォルダに以下の２つがある
・マクロを実行するマクロファイル
・「input」フォルダ
　フォルダ内に複数のcsvファイルがある
　ファイル名の末尾に日付が記載されている（例：***_2024-01-01.csv）

#要件
・フォルダ内のファイルに「Data」列をA列に追加し、全ての行に日付（例：20230101）を記載する
・マクロファイルのアクティブシートに対して、ファイルごとに行を追加しながら、全ての列のデータをコピーする
・マクロファイルを保存して閉じる
・「全ての処理が終了しました」のメッセージを出す


■VBA マクロ作成手順

■VBA マクロ実行手順

■Pythonコード生成事例
EALデータのようなデータ分析をする事例です

■Python コード作成手順
【プロンプト】
あなたは優秀なPythonのプログラマです。
以下の #要件 に合うPythonのコードを作成してください。

#前提条件
フォルダに以下の３つがある
・コードを実行するpyファイル
・「input」フォルダ
　フォルダ内に複数のcsvファイルがある
　ファイル名の先頭に号機名が記載されている（例：AB1_**.csv）
　ファイル名の末尾にファイル種類が記載されている（例：**_10.csv）
　「A」「B」列のデータがある
・「output」フォルダ

#要件
・「input」フォルダ内でファイル種類が「1」のデータを読み込む
・データに「id」列を追加し、行に号機を記載する
・データを全て結合する
・「id」「A」の昇順で並び替える
・「id」ごとの「B」の最大値を計算する
・縦軸が「id」で横軸が「A」の最大値の棒グラフを作成する
・「output」フォルダに「グラフの画像ファイル」と「データのcsvファイル」を保存する

■Python コード作成

■Python コード実行


■batファイル作成事例
ファイル操作と、Python実行処理を自動化する事例です


■batファイル コード生成手順
【プロンプト】
あなたは優秀なプログラマです。
以下の #要件 に合うbatファイルのコードを作成してください。

#前提条件
フォルダに以下の３つがある
・コードを実行するbatファイル
・「転送前フォルダ」フォルダ（日本語を含む絶対パス）
　フォルダ内に複数のcsvファイルがある
・「転送後フォルダ」フォルダ（日本語を含む絶対パス）
　フォルダ内にpyファイルがある

#要件
・「転送前フォルダ」と「転送後フォルダ」のパスを設定する
・「転送前フォルダ」内のcsvファイルの有無を確認し、存在する場合は全て「転送後フォルダ」にコピーする
・コピーが完了したら「転送前フォルダ」のファイルを削除する
・「転送後フォルダ」のpyファイルを実行する
・処理が終了したらプロンプト画面を閉じる

■batファイル作成手順

■batファイル実行手順